### Runnable 객체

**실행 가능 객체**

+ `Runnable`은 단일 입력에 대해 출력을 생성할 수 있는 **모든 객체**를 말한다. 

+ `ChatPromptTemplate` (프롬프트), `ChatGoogleGenerativeAI` (모델), `StrOutputParser` (파서) 등이 모두 `Runnable`이다.

**체인 구성**

+ `|` (파이프) 연산자를 사용하여 여러 `Runnable` 객체를 연결하여 하나의 복합적인 **체인**을 만들 수 있다. 

+ 연결된 체인 자체도 `Runnable` 객체로 취급된다.

**다양한 실행 방식**

+ `invoke()`, `stream()`, `batch()`, `ainvoke()`와 같은 메서드를 통해 단일 실행, 스트리밍, 병렬 처리 등 다양한 방식으로 작업을 수행할 수 있다.

### LCEL(LangChain Expression Language)

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

In [2]:
from langchain_core.prompts import PromptTemplate

template = "{language} 할 수 있어?"

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='{language} 할 수 있어?')

In [ ]:
prompt.invoke("한국말") # runnable

StringPromptValue(text='한국말 할 수 있어?')

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1, # 창의성 (0.0 ~ 2.0)
    google_api_key=gemini_api_key
)


In [6]:
model.invoke(prompt.invoke("한국말")) # runnable

AIMessage(content='네, 한국말 할 수 있습니다. 어떻게 도와드릴까요?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--f2063bae-f23b-4864-b6ce-179f6247ed9b-0', usage_metadata={'input_tokens': 7, 'output_tokens': 60, 'total_tokens': 67, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 47}})

In [ ]:
chain = prompt | model # chain 생성

In [8]:
input = {"language": "한국말"}

chain.invoke(input)

AIMessage(content='네, 한국말 할 수 있습니다. 어떻게 도와드릴까요?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--9a5ec96f-d9c4-4129-9946-9d6fca51f084-0', usage_metadata={'input_tokens': 7, 'output_tokens': 73, 'total_tokens': 80, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 60}})

### 스트리밍 출력

In [ ]:
template = "{topic}에 대해 아주 쉽게 설명해줘?"

prompt = PromptTemplate.from_template(template)

In [10]:
input = {"topic": "LangChain"}

In [11]:
chain = prompt | model

In [12]:
response = chain.stream(input)

In [13]:
next(response)

AIMessageChunk(content='네, LangChain에 대해 아주 쉽게 설명해 드릴게요!\n\n---\n\n### LangChain, LLM의 슈퍼히어로 도우미! 🦸\u200d♂️\n\n**1. LLM은 누구인가요? (똑똑한 뇌', additional_kwargs={}, response_metadata={'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--39845439-d2f1-4a76-9a9e-dbd1bf45476f', usage_metadata={'input_tokens': 10, 'output_tokens': 1417, 'total_tokens': 1427, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 1367}})

In [14]:
next(response)

AIMessageChunk(content=")**\n*   ChatGPT 같은 **LLM(거대 언어 모델)**은 정말 똑똑한 '뇌'라고 생각하면 돼요.\n*   말을 아주 잘하고, 글을 쓰고, 요약하고, 번역하는 등 언", additional_kwargs={}, response_metadata={'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--39845439-d2f1-4a76-9a9e-dbd1bf45476f', usage_metadata={'total_tokens': 51, 'output_token_details': {'reasoning': 0}, 'input_tokens': 0, 'input_token_details': {'cache_read': 0}, 'output_tokens': 51})

In [15]:
for token in response:
    print(token.content, end="", flush=True)

어와 관련된 일은 기가 막히게 잘하죠.
*   하지만 이 '뇌'는 혼자서는 할 수 없는 일들이 있어요. 예를 들면:
    *   "지금 날씨가 어때?" (실시간 정보 검색 불가)
    *   "내 이메일에서 중요한 내용만 요약해줘." (외부 데이터 접근 불가)
    *   "어제 나랑 나눴던 대화 내용 기억해?" (기억력 없음)
    *   "이 정보를 바탕으로 웹사이트에 글을 올려줘." (실제 행동 불가)

**2. LangChain은 무엇인가요? (뇌에 몸과 도구를 연결해주는 마법사)**
*   바로 이 똑똑한 '뇌'인 LLM이 **실제로 세상과 소통하고, 다양한 도구를 사용하고, 복잡한 작업을 수행할 수 있도록 도와주는 '다리'이자 '도구 상자' 같은 존재**가 LangChain이에요.
*   LangChain 덕분에 LLM은 단순한 '뇌'를 넘어, 실제 세상에서 유용한 일을 할 수 있는 '몸'과 '손발'을 얻게 되는 거죠.

**3. LangChain이 하는 일 (LLM을 더 똑똑하고 유용하게 만드는 방법)**

LangChain은 LLM에게 다음과 같은 능력을 부여해줘요:

*   **기억력 (Memory):**
    *   LLM이 이전 대화 내용을 기억하게 해줘요. 마치 우리가 친구와 대화할 때 지난 이야기를 기억하는 것처럼요.
    *   **예시:** "어제 물어봤던 그 주제에 대해 더 알려줘."

*   **외부 정보 연결 (Retrieval - 검색):**
    *   LLM이 인터넷 검색, 특정 문서(PDF, 워드 파일 등), 데이터베이스 등 **자신이 학습하지 않은 최신 정보나 특정 정보를 찾아보고 활용**할 수 있게 해줘요.
    *   **예시:** "오늘 환율이 얼마야?" 또는 "우리 회사 내부 규정집에 따르면 이 문제는 어떻게 처리해야 해?"

*   **도구 사용 (Tools & Agents - 도구와 에이전트):**
    *   LLM이 계산기, 웹 브라우저, 다른 프로그램(API) 등 **다양한 외부 도구를 